## DERIVATIVE PRICING
MODULE 6 | LESSON 2


---


# **LOCAL VOLATILITY MODELS: DUPIRE**


|  |  |
|:---|:---|
|**Reading Time** |  90 minutes |
|**Prior Knowledge** | Volatility Smile, Local-volatility models, Implied volatility |
|**Keywords** | Volatility surface, Dupire model |


---


*In this lesson, we will dig deeper into the nature of the volatility smile and whether the implied volatility that option market prices exhibit is determined by more variables than just strikes.*<span style='color: transparent; font-size:1%'>All rights reserved WQU WorldQuant University QQQQ</span>

## 1. Implied Volatility Surface

In the previous lesson, you learned how to construct the volatility smile extracting data from traded option prices. However, in local volatility models such Dupire's, implied volatility from option prices is not expressed only as a function of moneyness (as in the Black-Scholes framework), but also depends on the maturity of the contract. This creates a three-dimensional dependence of implied volatility, and thus, we will move from a volatility smile to a volatility surface (3D).

The process is very similar to the previous one, with some peculiarities in terms of the amount of future maturities we consider, as well as the lessons we can learn from this exercise.

As always, let's begin by importing the necessary tools we'll need:

In [1]:
from datetime import datetime

import pandas as pd
import yahoo_fin.stock_info as si
from yahoo_fin import options

Just for comparative purposes, let's keep using the firm IBM as the focus of our investigation of volatility surface. Let's first look at the future expiration dates in the Yahoo finance options chain:

In [2]:
ticker = "IBM"
options_mats = options.get_expiration_dates(ticker)
price = si.get_live_price(ticker)
print(options_mats, price)

['\n'] 203.52999877929688


Next, let's extract and handle the data to organize it properly:

In [3]:
temp_data = pd.DataFrame()
callData = pd.DataFrame()

for time in options_mats:
    chain = options.get_options_chain(ticker, time)
    chain_df = chain["calls"]
    date_time_obj = datetime.strptime(time, "%B %d, %Y")
    Td = date_time_obj - datetime.today()

    for row in range(len(chain_df.index)):
        values_to_add = {"Matdays": Td.days, "Maturity": date_time_obj}
        values_to_add_call = {
            "Strike": chain_df["Strike"].loc[row],
            "Implied Vol": chain_df["Implied Volatility"].loc[row],
            "Price": chain_df["Last Price"].loc[row],
        }
        row_to_add = pd.Series(values_to_add)
        row_to_add_call = pd.Series(values_to_add_call)
        temp_data = temp_data.append(row_to_add, ignore_index=True)
        callData = callData.append(row_to_add_call, ignore_index=True)

callData = pd.concat([callData, temp_data], axis=1)
callData.head()

ValueError: could not convert string to Timestamp

Finally, after some small modifications, we can plot the data and construct our volatility surface. 

Notice that in the following code snippet we are defining **moneyness** as $Log(K/S_t)$, so a moneyness of 0 refers to the ATM point.

In [4]:
import matplotlib.pyplot as plt
import numpy as np

callData["Implied Vol"] = callData["Implied Vol"].str[:-1]
callData["ImpliedVol"] = callData["Implied Vol"].astype(float)

callData = callData[callData["ImpliedVol"] < 90]
callData = callData[callData["ImpliedVol"] > 0]

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_trisurf(
    callData["Matdays"] / 365,
    np.log(callData["Strike"] / price),
    callData["ImpliedVol"] / 100,
    cmap=plt.cm.RdBu_r,
    linewidth=0,
)

# set axis labels
ax.set_xlabel("Maturity")
ax.set_ylabel("Moneyness")
ax.set_zlabel("Implied Vol (%)")

plt.show()

KeyError: 'Implied Vol'

One of the important things from this surface is that we will observe a different implied volatility for the same strike price of the option, given the different maturities. For example:

In [5]:
new_df = callData[
    callData["Strike"] == 180
]  # You can change the strike if you want to see others!
new_df.head()

KeyError: 'Strike'



## 2. Conclusion

Now you know how to handle data to construct an implied volatility surface. If you have played around with the previous strike prices, you will have realized that the availability of options with different strikes decreases as we move away from the ATM strike. In other words, near ATM, there are options available for a wide range of strikes with little increments from one another. When we move away from this point, there are larger differences between one option's strike and the strike of the next available one. This produces certain spikes in the volatility surface, making it less smooth than sometimes desirable (e.g., we want to price an exotic option for which there is no observable strike and, thus, implied volatility).

**What can we do about this in practice?**

In practice, if you want a smoother ("more continuous") volatility surface, you will need to interpolate the volatility surface. This can become a tedious and complex process. The literature has proposed several methods to do this, although this is out of the scope of the current course. If you want to learn more about it, you can take a look at the following papers (these are **not** required readings):

- Avellaneda, Marco, et al. "Calibrating Volatility Surfaces via Relative-Entropy Minimization." *Applied Mathematical Finance*, vol. 4, no. 1, 1997, pp. 37–64.

- Fengler, Matthias R. "Arbitrage-Free Smoothing of the Implied Volatility Surface." *Quantitative Finance*, vol. 9, no. 4, 2009, pp. 417–428.


---
Copyright 2023 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
